In [4]:
from tkinter import Tk, filedialog
import pandas as pd

# Masquer la fenêtre principale Tkinter
root = Tk()
root.withdraw()

# Forcer la fenêtre à être au premier plan
root.attributes('-topmost', True)
root.update()

# Ouvrir la fenêtre de sélection
fichier = filedialog.askopenfilename(
    title="Choisissez le fichier à analyser",
    filetypes=[("Fichiers CSV", "*.csv"), ("Fichiers Excel", "*.xlsx *.xls"), ("Tous les fichiers", "*.*")]
)

# Lecture du fichier sélectionné avec pandas
df = pd.read_csv(fichier)

FileNotFoundError: [Errno 2] No such file or directory: ''

In [2]:
df.columns

Index(['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',
       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Description',
       'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone',
       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',
       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
       'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight'],
      dtype='object')

## Echantillonnage

### Vérification de la couverture

In [3]:
pd.crosstab(df['State'], df['month'])

KeyError: 'month'

### échantillonnage stratifié State × Month

#### prendre 10% de chaque (State, month)

In [4]:
sample_frac = 0.10  # 10%

df_sample = (
    df
    .groupby(['State', 'month'], group_keys=False)
    .apply(lambda x: x.sample(
        frac=sample_frac,
        random_state=42
    ) if len(x) > 1 else x)
)


#### Tous les États ≈ même proportion relative que le dataset initial

In [5]:
df['State'].value_counts(normalize=True)

State
CA    0.226100
FL    0.113965
TX    0.075706
SC    0.049654
NY    0.045148
NC    0.043910
VA    0.039397
PA    0.038490
MN    0.024934
OR    0.022847
GA    0.021964
IL    0.021945
AZ    0.021819
TN    0.021746
MI    0.021069
NJ    0.018253
LA    0.018245
MD    0.018240
OH    0.015338
WA    0.014003
AL    0.013114
UT    0.012582
CO    0.011792
OK    0.010357
MO    0.010036
CT    0.009223
IN    0.008726
MA    0.008059
WI    0.004505
KY    0.004188
NE    0.003704
MT    0.003456
IA    0.003396
AR    0.002958
NV    0.002789
KS    0.002687
DC    0.002414
RI    0.002206
MS    0.001968
DE    0.001790
WV    0.001768
ID    0.001456
NM    0.001338
NH    0.001322
ND    0.000451
WY    0.000439
ME    0.000348
VT    0.000120
SD    0.000036
Name: proportion, dtype: float64

In [5]:
df_sample['State'].value_counts(normalize=True)

State
CA    0.226100
FL    0.113965
TX    0.075706
SC    0.049654
NY    0.045148
NC    0.043910
VA    0.039397
PA    0.038490
MN    0.024934
OR    0.022847
GA    0.021964
IL    0.021945
AZ    0.021819
TN    0.021746
MI    0.021069
NJ    0.018253
LA    0.018245
MD    0.018240
OH    0.015338
WA    0.014003
AL    0.013114
UT    0.012582
CO    0.011792
OK    0.010357
MO    0.010036
CT    0.009223
IN    0.008726
MA    0.008059
WI    0.004505
KY    0.004188
NE    0.003704
MT    0.003456
IA    0.003396
AR    0.002958
NV    0.002789
KS    0.002687
DC    0.002414
RI    0.002206
MS    0.001968
DE    0.001790
WV    0.001768
ID    0.001456
NM    0.001338
NH    0.001322
ND    0.000451
WY    0.000439
ME    0.000348
VT    0.000120
SD    0.000036
Name: proportion, dtype: float64

#### Vérifier que les 12 mois apparaissent

In [6]:
df_sample['month'].value_counts().sort_index()

month
1     7480
2     6558
3     5527
4     5850
5     5562
6     5693
7     5102
8     5975
9     6480
10    6723
11    7564
12    8423
Name: count, dtype: int64

#### Vérifier que aucune cellule vide (ou très peu)

In [7]:
pd.crosstab(df_sample['State'], df_sample['month'])

month,1,2,3,4,5,6,7,8,9,10,11,12
State,,,,,,,,,,,,
AL,91,77,82,84,81,82,73,81,80,87,95,97
AR,27,21,15,15,14,16,13,15,19,18,26,28
AZ,160,134,128,150,138,120,104,139,140,135,163,168
CA,1681,1570,1392,1391,1257,1310,1082,1243,1476,1471,1647,1875
CO,89,82,75,70,67,70,60,73,70,78,80,94
CT,61,56,45,51,48,59,55,55,65,68,75,70
DC,17,13,12,17,15,15,12,12,18,16,19,20
DE,10,10,10,12,12,13,12,11,11,12,13,12
FL,893,720,630,717,650,579,529,621,719,774,921,1014


Un échantillonnage stratifié par État et par mois a été appliqué afin d’assurer une représentation équitable des régions et des variations saisonnières dans l’analyse statistique.

In [8]:
df_sample.shape

(76937, 83)

## Statistique Inférentielle

### Comparer Severity entre 2 groupes : pluie vs sec (Test t ou Mann-Whitney)

Variable continue : Severity

Deux groupes : Rain vs Dry (Clear, Cloudy, Fog)

On teste :

- H0: la moyenne de Severity est la meme dans les 2 groupes
- H1​:les moyennes sont différentes

Interprétation:

- t_stat → mesure la différence entre les moyennes en termes d’écart-type.
- p_val → probabilité d’obtenir cette différence si H0 est vraie.

p < 0.05 → la météo influence significativement la gravité.

In [12]:
from scipy.stats import ttest_ind, mannwhitneyu

# Séparer les groupes
rain_severity = df_sample[df_sample['Weather_Category']=='Rain']['Severity']
dry_severity = df_sample[df_sample['Weather_Category'].isin(['Clear','Cloudy','Fog'])]['Severity']

# Test t 2 échantillons
t_stat, p_val = ttest_ind(rain_severity, dry_severity, equal_var=False)
print(f"T-test (Rain vs Dry): t={t_stat:.3f}, p={p_val:.5f}")

# Test non-paramétrique (si la distribution n'est pas normale)
u_stat, p_val_u = mannwhitneyu(rain_severity, dry_severity)
print(f"Mann-Whitney (Rain vs Dry): U={u_stat}, p={p_val_u:.5f}")

T-test (Rain vs Dry): t=19.488, p=0.00000
Mann-Whitney (Rain vs Dry): U=18539246198.5, p=0.00000


### ANOVA pour comparer Severity entre plusieurs catégories (Weather_Category)

ANOVA = Analysis Of Variance répond à la question :

Les différences observées entre groupes sont-elles réelles ou dues au hasard ?

sum_sq (Somme des carrés):

- Mesure la variabilité
- Plus elle est grande → plus la variable explique la variation de Severity

df (degrés de liberté):

- Liés au nombre de catégories météo

F (statistique F):
- F élevé → groupes très différents
- F proche de 1 → groupes similaires

PR(>F) (p-value):
- < 0.05 → Différence significative
- < 0.01 → Très significative
- ≥ 0.05 → Pas de preuve de différence


In [14]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('Severity ~ C(Weather_Category)', data=df_sample).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                            sum_sq        df           F  PR(>F)
C(Weather_Category)     594.843584       8.0  314.207422     0.0
Residual             182058.871414  769335.0         NaN     NaN


### Corrélation entre variables continues

#### Pearson mesure la relation linéaire entre deux variables continues.

Quand la vitesse du vent augmente, est-ce que la distance de l’accident augmente ou diminue de façon linéaire ?

Résultats retournés:
- r → coefficient de corrélation (entre -1 et +1)
- p → p-value (significativité statistique)

r ≈ 0 → Pas de relation linéaire

p = 0.0000 → le lien n’est pas dû au hasard

#### Spearman mesure une relation monotone (pas forcément linéaire), basée sur le classement des valeurs.
Quand la visibilité diminue, est-ce que la sévérité tend à augmenter globalement, même si la relation n’est pas linéaire ?

Résultats retournés:
- rho → coefficient de corrélation monotone
- p_s → significativité

rho = -0.09 → négatif dnc quand visibilité ↓ → sévérité ↑

In [18]:
from scipy.stats import spearmanr, pearsonr

subset_s = df_sample[['Severity', 'Visibility(mi)']].dropna()
subset_p = df_sample[['Wind_Speed(mph)', 'Distance(mi)']].dropna()


# Pearson pour linéaire
r, p = pearsonr(
    subset_p['Wind_Speed(mph)'],
    subset_p['Distance(mi)']
)
print(f"Pearson Wind vs Distance: r={r:.3f}, p={p:.5f}")


# Spearman pour rangs
rho, p_s = spearmanr(
    subset_s['Severity'],
    subset_s['Visibility(mi)']
)
print(f"Spearman Severity vs Visibility: rho={rho:.3f}, p={p_s:.5f}")


Pearson Wind vs Distance: r=0.011, p=0.00000
Spearman Severity vs Visibility: rho=-0.009, p=0.00000


### Régression linéaire simple et multiple

Comment lire lm2.summary() :

| Élément            | Signification           |
| ------------------ | ----------------------- |
| R-squared          | % de variance expliquée |
| Adj. R-squared     | R² pénalisé             |
| F-statistic        | Significativité globale |
| Prob (F-statistic) | p-value globale         |


In [20]:
import statsmodels.formula.api as smf

# Multiple : Severity ~ Weather + Visibility + Wind
lm2 = smf.ols(
    'Severity ~ C(Weather_Category) + Q("Visibility(mi)") + Q("Wind_Speed(mph)")',
    data=df_sample
).fit()

print(lm2.summary())

                            OLS Regression Results                            
Dep. Variable:               Severity   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     277.2
Date:                Thu, 15 Jan 2026   Prob (F-statistic):               0.00
Time:                        15:47:13   Log-Likelihood:            -5.3180e+05
No. Observations:              763337   AIC:                         1.064e+06
Df Residuals:                  763326   BIC:                         1.064e+06
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

## Insights

In [27]:
# Masquer la fenêtre principale Tkinter
root = Tk()
root.withdraw()

# Forcer la fenêtre au premier plan
root.attributes('-topmost', True)
root.update()

# Fenêtre pour choisir où sauvegarder le fichier
chemin_fichier = filedialog.asksaveasfilename(
    title="Choisissez l'emplacement pour sauvegarder le fichier",
    defaultextension=".csv",
    filetypes=[("Fichier CSV", "*.csv"), ("Tous les fichiers", "*.*")]
)

# Sauvegarde du fichier si un chemin est sélectionné
if chemin_fichier:
    df_sample.to_csv(chemin_fichier, index=False)
    print("Fichier sauvegardé avec succès :", chemin_fichier)
else:
    print("Sauvegarde annulée")


Fichier sauvegardé avec succès : C:/Users/pc/Downloads/US_Accidents/US_Accidents_sample.csv


<style>
T1 {
    display: inline-block;
    background: #fde2d2;
    color: #c95305;
    padding: 10px 16px;
    border-radius: 8px;
    font-size: 19px;
    font-weight: 900;
    font-family: "Segoe UI", Roboto, Arial, sans-serif;
    box-shadow: 0 4px 10px rgba(0,0,0,0.12);
    letter-spacing: 0.2px;
    border: 1px solid rgba(255,255,255,0.15);
}
</style>

<T1>🚧 Pour les donnees infrastructurelles</T1>

In [ ]:
# Les caractéristiques de l’infrastructure routière influencent-elles la gravité des accidents ?

In [ ]:
infra_cols = [
    "Infrastructure_Count", "Infrastructure_Density", "Has_Infrastructure",
    "Has_Safety_Device", "Safety_Device_Count", "Has_Conflict_Point",
    "Conflict_Point_Count", "Junction_Type", "Infrastructure_Risk_Score", "Protection_Index"
]

df[infra_cols].info()
df[infra_cols].describe()